# Tugas 5 — Information Retrieval

## Tujuan

Pada tugas ini dilakukan implementasi konsep dasar Information Retrieval,
meliputi pembuatan matrix dan array, perhitungan TF-IDF secara manual,
pembuatan search engine menggunakan TF-IDF dan Cosine Similarity, serta
pembuatan dictionary teks, posting list, dan inverted index.

In [12]:
import numpy as np
import pandas as pd 
import math


In [13]:
documents = {
    "D1": "saya belajar python",
    "D2": "saya belajar data",
    "D3": "python untuk data"
}

for doc_id, text in documents.items():
    print(f"{doc_id}: {text}")

D1: saya belajar python
D2: saya belajar data
D3: python untuk data


## 1. Matrix 1.000.000 × 1.000.000

S### Simulasi Matriks 1.000.000 × 1.000.000

Matriks berukuran **1.000.000 × 1.000.000** berarti memiliki satu juta baris dan satu juta kolom. Jumlah seluruh elemen pada matriks tersebut dapat dihitung sebagai:

**1.000.000 × 1.000.000 = 1.000.000.000.000 elemen**

atau setara dengan **1 triliun elemen**.

Jika setiap elemen disimpan menggunakan tipe data `float64`, maka setiap elemen membutuhkan ruang penyimpanan sebesar **8 byte**. Dengan demikian, estimasi kebutuhan memori adalah:

**1.000.000.000.000 × 8 byte = 8.000.000.000.000 byte**

atau sekitar **7,28 TB** memori.

Ukuran tersebut jauh lebih besar dibandingkan kapasitas RAM komputer pada umumnya. Oleh karena itu, matriks **1.000.000 × 1.000.000 tidak dibuat secara langsung dalam bentuk dense matrix**. Pada praktikum ini hanya dilakukan **simulasi perhitungan ukuran, jumlah elemen, dan kebutuhan memorinya**.

Konsep ini juga menunjukkan alasan mengapa pengolahan data berskala besar, seperti pada **Information Retrieval dan TF-IDF**, sering menggunakan **sparse matrix**, karena sebagian besar nilai dalam matriks biasanya bernilai **0** sehingga penyimpanan dapat dilakukan dengan lebih efisien.

In [14]:
rows = 1_000_000
cols = 1_000_000

total_elements = rows * cols

print("Jumlah baris      :", rows)
print("Jumlah kolom      :", cols)
print("Ukuran matriks    :", f"{rows:,} x {cols:,}")
print("Jumlah elemen     :", f"{total_elements:,}")

Jumlah baris      : 1000000
Jumlah kolom      : 1000000
Ukuran matriks    : 1,000,000 x 1,000,000
Jumlah elemen     : 1,000,000,000,000


In [15]:
bytes_per_float64 = 8
memory_bytes = total_elements * bytes_per_float64
memory_tb = memory_bytes / (1024**4)

print("Estimasi memori float64:")
print(f"{memory_tb:,.2f} TB")

Estimasi memori float64:
7.28 TB


## 2. Matrix dan Array

Matrix digunakan untuk merepresentasikan hubungan antara document dan term.
Array digunakan untuk menyimpan kumpulan term yang digunakan dalam proses
Information Retrieval.

In [16]:
import numpy as np
import pandas as pd
import math

documents = {
    "D1": "saya belajar python",
    "D2": "saya belajar data",
    "D3": "python untuk data"
}

terms = ["belajar", "data", "python", "saya", "untuk"]

# Term-Document Matrix
matrix = []

for doc_id, text in documents.items():
    words = text.lower().split()
    row = [1 if term in words else 0 for term in terms]
    matrix.append(row)

matrix = np.array(matrix)

matrix_df = pd.DataFrame(
    matrix,
    index=documents.keys(),
    columns=terms
)

print("MATRIX:")
display(matrix_df)

# Array
term_array = np.array(terms)

print("\nARRAY:")
print(term_array)

MATRIX:


,belajar,data,python,saya,untuk
D1,1,0,1,1,0
D2,1,1,0,1,0
D3,0,1,1,0,1



ARRAY:
['belajar' 'data' 'python' 'saya' 'untuk']


## 3. Perhitungan TF-IDF 5 Kata Secara Manual

TF (Term Frequency) menunjukkan frekuensi kemunculan term dalam document.

IDF (Inverse Document Frequency) menunjukkan tingkat kepentingan term
berdasarkan jumlah document yang mengandung term tersebut.

TF-IDF diperoleh dari perkalian TF dan IDF.

In [17]:
# Hitung TF
tf_matrix = []

for doc_id, text in documents.items():
    words = text.lower().split()
    total_words = len(words)
    
    row = []
    for term in terms:
        tf = words.count(term) / total_words
        row.append(tf)
    
    tf_matrix.append(row)

tf_df = pd.DataFrame(
    tf_matrix,
    index=documents.keys(),
    columns=terms
)

print("TF:")
display(tf_df)

# Document Frequency
df_values = []

for term in terms:
    df = sum(term in text.lower().split() for text in documents.values())
    df_values.append(df)

df_df = pd.DataFrame({
    "Term": terms,
    "DF": df_values
})

print("Document Frequency:")
display(df_df)

# IDF
N = len(documents)

idf_values = [
    math.log10(N / df)
    for df in df_values
]

idf_df = pd.DataFrame({
    "Term": terms,
    "IDF": idf_values
})

print("IDF:")
display(idf_df)

# TF-IDF
tfidf_matrix = tf_matrix * np.array(idf_values)

tfidf_df = pd.DataFrame(
    tfidf_matrix,
    index=documents.keys(),
    columns=terms
)

print("TF-IDF:")
display(tfidf_df)

TF:


,belajar,data,python,saya,untuk
D1,0.333333,0.000000,0.333333,0.333333,0.000000
D2,0.333333,0.333333,0.000000,0.333333,0.000000
D3,0.000000,0.333333,0.333333,0.000000,0.333333


Document Frequency:


,Term,DF
0,belajar,2
1,data,2
2,python,2
3,saya,2
4,untuk,1


IDF:


,Term,IDF
0,belajar,0.176091
1,data,0.176091
2,python,0.176091
3,saya,0.176091
4,untuk,0.477121


TF-IDF:


,belajar,data,python,saya,untuk
D1,0.058697,0.000000,0.058697,0.058697,0.00000
D2,0.058697,0.058697,0.000000,0.058697,0.00000
D3,0.000000,0.058697,0.058697,0.000000,0.15904


## 4. Search Engine Menggunakan TF-IDF dan Cosine Similarity

Search engine digunakan untuk mencari document yang paling relevan terhadap
query pengguna.

Relevansi dihitung menggunakan Cosine Similarity antara vector query dan
vector TF-IDF document.

In [18]:
def cosine_similarity(vec1, vec2):
    numerator = np.dot(vec1, vec2)
    
    denominator = (
        np.linalg.norm(vec1) * np.linalg.norm(vec2)
    )
    
    if denominator == 0:
        return 0
    
    return numerator / denominator


def search_engine(query):
    query_words = query.lower().split()
    
    # TF query
    query_tf = np.array([
        query_words.count(term) / len(query_words)
        for term in terms
    ])
    
    # TF-IDF query
    query_tfidf = query_tf * np.array(idf_values)
    
    results = []
    
    for i, doc_id in enumerate(documents.keys()):
        score = cosine_similarity(
            query_tfidf,
            tfidf_matrix[i]
        )
        
        results.append((doc_id, score))
    
    results.sort(key=lambda x: x[1], reverse=True)
    
    return results

In [19]:
query = "belajar python"

results = search_engine(query)

print("Query:", query)
print("\nHasil Search:")

for doc_id, score in results:
    print(f"{doc_id}: {score:.4f} - {documents[doc_id]}")

Query: belajar python

Hasil Search:
D1: 0.8165 - saya belajar python
D2: 0.4082 - saya belajar data
D3: 0.2314 - python untuk data


## 5. Dictionary Teks

Dictionary teks digunakan untuk menyimpan informasi document dan term
yang terdapat di dalamnya.

In [20]:
text_dictionary = {}

for doc_id, text in documents.items():
    text_dictionary[doc_id] = text.lower().split()

print("TEXT DICTIONARY:")

for doc_id, terms_doc in text_dictionary.items():
    print(doc_id, ":", terms_doc)

TEXT DICTIONARY:
D1 : ['saya', 'belajar', 'python']
D2 : ['saya', 'belajar', 'data']
D3 : ['python', 'untuk', 'data']


## 6. Posting List

Posting list berisi daftar document ID tempat suatu term ditemukan.

In [21]:
posting_list = {}

for term in terms:
    posting_list[term] = []
    
    for doc_id, text in documents.items():
        if term in text.lower().split():
            posting_list[term].append(doc_id)

print("POSTING LIST:")

for term, postings in posting_list.items():
    print(f"{term}: {postings}")

POSTING LIST:
belajar: ['D1', 'D2']
data: ['D2', 'D3']
python: ['D1', 'D3']
saya: ['D1', 'D2']
untuk: ['D3']


## 7. Inverted Index

Inverted index merupakan struktur data yang memetakan term ke document
tempat term tersebut muncul.

In [22]:
inverted_index = {}

for term in terms:
    inverted_index[term] = posting_list[term]

print("INVERTED INDEX:")

for term, postings in inverted_index.items():
    print(f"{term} -> {postings}")

INVERTED INDEX:
belajar -> ['D1', 'D2']
data -> ['D2', 'D3']
python -> ['D1', 'D3']
saya -> ['D1', 'D2']
untuk -> ['D3']


## 8. Arsitektur Information Retrieval

Alur sistem yang digunakan adalah:

Document
↓
Tokenisasi / Term
↓
Term-Document Matrix
↓
TF dan IDF
↓
TF-IDF
↓
Query User
↓
TF-IDF Query
↓
Cosine Similarity
↓
Ranking Document
↓
Search Result

Untuk mempercepat pencarian term, digunakan struktur Dictionary,
Posting List, dan Inverted Index.
Selain itu, dilakukan simulasi **matrix 1.000.000 × 1.000.000** untuk menghitung jumlah elemen dan kebutuhan memorinya tanpa membuat matrix secara langsung.

## 9. Kesimpulan

## Kesimpulan

Pada tugas ini telah dilakukan proses pembentukan Term-Document Matrix, 
perhitungan TF-IDF, serta penerapan Cosine Similarity untuk membangun 
sederhana search engine. Selain itu, digunakan Dictionary, Posting List, 
dan Inverted Index untuk membantu pencarian term.

Selain itu, dilakukan simulasi **matrix 1.000.000 × 1.000.000** untuk 
menghitung jumlah elemen dan kebutuhan memorinya tanpa membuat matrix 
secara langsung.